# Proyecto guiado: construir un RAG desde cero con `Renting.pdf`

En este notebook vamos a construir un **RAG completo**, pero de forma deliberadamente transparente: cada etapa quedará visible y podremos inspeccionarla antes de pasar a la siguiente.

Usaremos como base de conocimiento el documento `Renting.pdf`, que contiene las condiciones de una campaña de **Renting Tecnológico** y sus incentivos. fileciteturn21file0

## Objetivo

Entender y ejecutar paso a paso:

```text
                 OFFLINE / PREPROCESAMIENTO
Renting.pdf → Documentos → Chunks → Embeddings → ChromaDB
                                                   │
                                                   ▼
                 ONLINE / CONSULTA
Pregunta → Embedding de pregunta → Similarity Search → Top-K
                                                   ↓
                                                Contexto
                                                   ↓
                                                 Prompt
                                                   ↓
                                                Gemini
                                                   ↓
                                               Respuesta
```

**Idea clave:** los embeddings de los documentos se calculan durante el procesamiento offline. Cuando llega una pregunta, calculamos el embedding de la pregunta y consultamos ChromaDB.


## 1. Preparar el entorno

Si estás utilizando el entorno `rag` del curso, probablemente ya tengas estas librerías.

Si falta alguna, ejecuta la celda siguiente.


In [1]:
# Ejecuta esta celda solo si necesitas instalar dependencias.
# %pip install pypdf chromadb google-genai langchain-community langchain-text-splitters python-dotenv


## 2. Importaciones

Utilizaremos:

- `PyPDFLoader` para convertir el PDF en documentos.
- `RecursiveCharacterTextSplitter` para hacer chunking.
- `google.genai` para embeddings y generación.
- `chromadb` para almacenar y recuperar los vectores.
- `dotenv` para leer `GEMINI_API_KEY`.


In [1]:
import os
from pathlib import Path

import chromadb
from dotenv import load_dotenv
from google import genai
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


C:\Users\Rubén\AppData\Local\Temp\ipykernel_26644\1308796175.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
c:\Users\Rubén\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. Configuración

El notebook es **independiente** del proyecto Sprint 10.

Coloca `Renting.pdf` en la misma carpeta que este notebook.

Usaremos:

- `gemini-embedding-2` para representar textos como vectores.
- `gemini-3.1-flash-lite` para generar la respuesta.
- ChromaDB persistente en `./chroma_renting_db`.
- Una colección llamada `renting_documentos`.
- `TOP_K = 3`.

> **Importante:** el modelo utilizado para el embedding de la pregunta debe ser el mismo que utilizamos para los documentos.


In [4]:
load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")

if not api_key:
    raise ValueError(
        "No se encontró GEMINI_API_KEY. "
        "Configúrala como variable de entorno o en un archivo .env."
    )

client = genai.Client(api_key=api_key)

PDF_PATH = Path("Renting.pdf")
CHROMA_DIR = Path("chroma_renting_db")

EMBEDDING_MODEL = "gemini-embedding-2"
GEMINI_MODEL = "gemini-3.1-flash-lite"

CHUNK_SIZE = 1500       # caracteres
CHUNK_OVERLAP = 200     # caracteres
TOP_K = 3

print("Configuración preparada")
print("PDF:", PDF_PATH.resolve())
print("Embedding model:", EMBEDDING_MODEL)
print("Generation model:", GEMINI_MODEL)


Configuración preparada
PDF: C:\Users\Rubén\Documents\01-Estudios\Bootcamp IA\00 - Repos\02 - Camp AI\01 - The Bridge\05_RAG_Engineering\Sprint_10\Practica_live_review\EJERCICIO_LR_2\Renting.pdf
Embedding model: gemini-embedding-2
Generation model: gemini-3.1-flash-lite


## 4. Cargar el documento

Primero transformamos el PDF en objetos `Document`.

Cada documento conserva:

- `page_content`: el texto.
- `metadata`: información sobre el origen, como la página.

Todavía **no** tenemos embeddings ni ChromaDB.


In [5]:
if not PDF_PATH.exists():
    raise FileNotFoundError(
        f"No encuentro {PDF_PATH}. "
        "Coloca Renting.pdf en la misma carpeta que el notebook."
    )

loader = PyPDFLoader(str(PDF_PATH))
documents = loader.load()

print("Páginas/documentos cargados:", len(documents))
print("Metadata de la primera página:", documents[0].metadata)
print()
print(documents[0].page_content[:2000])


Páginas/documentos cargados: 4
Metadata de la primera página: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-09-01T16:26:47+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-09-01T16:26:47+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'Renting.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1'}

Renting:
*Renting ofrecido por Banco Iberico. Operación de renting sujeta a previa aprobación por parte del Banco. Oferta válida en
Península y Baleares hasta el 30/10/2026. Oferta no válida para Ceuta y Melilla.
**Renting ofrecido por Banco Iberico. Renta mensual del iPhone 17e 256 GB Sin Seguro 20,99€ €/mes (IVA incluido). Se
recibirá una bonificación de 20,99 € netos mensuales (tras aplicar la retención según normativa fiscal vigente, actualmente
el 19 %) por la contratación de un renting tecnológico a 36 meses para personas físicas que (1) domicilien por primera vez
su nómi

### ¿Qué acabamos de hacer?

```text
Renting.pdf
    ↓
PyPDFLoader
    ↓
list[Document]
```

Cada `Document` representa aquí una página del PDF.

La base de conocimiento todavía **no está preparada para una búsqueda semántica**. Antes debemos decidir cómo dividir el contenido.


## 5. Chunking

Un documento completo puede ser demasiado grande para recuperarlo como una única unidad.

Por eso lo dividimos en **chunks**.

En este ejemplo usamos:

- `chunk_size = 1500` **caracteres**
- `chunk_overlap = 200` **caracteres**

Esto es importante: `RecursiveCharacterTextSplitter` mide estos parámetros en **caracteres**, no en tokens.

```text
Documento
─────────────────────────────────────────────
        Chunk 1
              ────────────────
                    Chunk 2
                          ────────────────
                                Chunk 3
```

El `overlap` permite conservar continuidad entre chunks consecutivos.


In [6]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = splitter.split_documents(documents)

print("Documentos originales:", len(documents))
print("Chunks generados:", len(chunks))


Documentos originales: 4
Chunks generados: 13


## 6. Inspeccionar los chunks

Antes de seguir, **hay que mirar qué hemos creado**.

Esta es una práctica importante en RAG: un retrieval malo muchas veces empieza con un chunking malo.


In [7]:
for i, chunk in enumerate(chunks[:5]):
    print("=" * 80)
    print(f"CHUNK #{i}")
    print("Metadata:", chunk.metadata)
    print("Caracteres:", len(chunk.page_content))
    print()
    print(chunk.page_content[:1000])


CHUNK #0
Metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-09-01T16:26:47+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-09-01T16:26:47+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'Renting.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1'}
Caracteres: 1433

Renting:
*Renting ofrecido por Banco Iberico. Operación de renting sujeta a previa aprobación por parte del Banco. Oferta válida en
Península y Baleares hasta el 30/10/2026. Oferta no válida para Ceuta y Melilla.
**Renting ofrecido por Banco Iberico. Renta mensual del iPhone 17e 256 GB Sin Seguro 20,99€ €/mes (IVA incluido). Se
recibirá una bonificación de 20,99 € netos mensuales (tras aplicar la retención según normativa fiscal vigente, actualmente
el 19 %) por la contratación de un renting tecnológico a 36 meses para personas físicas que (1) domicilien por primera vez
su nómina o pensión superior a 1.

### Pregunta para el alumno

Mira los chunks anteriores y responde:

1. ¿Cada chunk contiene una idea suficientemente coherente?
2. ¿Hay frases cortadas?
3. ¿Qué ocurre en las zonas de overlap?
4. ¿Cambiarías `CHUNK_SIZE` o `CHUNK_OVERLAP`?

No existe un valor universalmente correcto. El chunking se valida con el corpus y con las preguntas que esperamos recibir.


## 7. Preparar metadata e identificadores

La metadata nos permitirá saber **de dónde salió la información recuperada**.

En este proyecto nos interesa especialmente:

- página del PDF;
- índice del chunk;
- nombre del archivo.


In [8]:
chunks_preparados = []

for i, chunk in enumerate(chunks):
    metadata = dict(chunk.metadata)
    metadata["source"] = PDF_PATH.name
    metadata["chunk_index"] = i

    chunks_preparados.append(
        {
            "id": f"chunk_{i}",
            "text": chunk.page_content,
            "metadata": metadata,
        }
    )

print("Chunks preparados:", len(chunks_preparados))
print(chunks_preparados[0])


Chunks preparados: 13
{'id': 'chunk_0', 'text': 'Renting:\n*Renting ofrecido por Banco Iberico. Operación de renting sujeta a previa aprobación por parte del Banco. Oferta válida en\nPenínsula y Baleares hasta el 30/10/2026. Oferta no válida para Ceuta y Melilla.\n**Renting ofrecido por Banco Iberico. Renta mensual del iPhone 17e 256 GB Sin Seguro 20,99€ €/mes (IVA incluido). Se\nrecibirá una bonificación de 20,99 € netos mensuales (tras aplicar la retención según normativa fiscal vigente, actualmente\nel 19 %) por la contratación de un renting tecnológico a 36 meses para personas físicas que (1) domicilien por primera vez\nsu nómina o pensión superior a 1.200 € o cuota de autónomos o mutualidad y (2) la mantengan junto con la domiciliación\nde dos recibos mensuales, (3) un movimiento mensual de tarjeta de crédito o saldo en cuenta igual o superior a 1.000 €\ntodos los días del mes y (4) tengan Bizum activo en Banco Iberico. Se recibirá una bonificación de 27,99 € netos mensuales\n(tra

## 8. Crear los embeddings

Ahora transformamos cada chunk de texto en un vector numérico.

```text
"texto del chunk"
       ↓
modelo de embeddings
       ↓
[0.012, -0.084, 0.231, ...]
```

Estos números representan semánticamente el contenido del texto.

Vamos a hacerlo por lotes para reducir el número de llamadas a la API.


In [9]:
def embed_texts(texts):
    vectores = []

    for i, text in enumerate(texts, start=1):

        response = client.models.embed_content(
            model=EMBEDDING_MODEL,
            contents=text,
        )

        vectores.append(response.embeddings[0].values)

        print(f"Embedding: {i}/{len(texts)}")

    return vectores


In [10]:
texts = [item["text"] for item in chunks_preparados]

embeddings = embed_texts(texts)

print()
print("Número de embeddings:", len(embeddings))
print("Dimensión del vector:", len(embeddings[0]))
print("Primeros valores:", embeddings[0][:10])


Embedding: 1/13
Embedding: 2/13
Embedding: 3/13
Embedding: 4/13
Embedding: 5/13
Embedding: 6/13
Embedding: 7/13
Embedding: 8/13
Embedding: 9/13
Embedding: 10/13
Embedding: 11/13
Embedding: 12/13
Embedding: 13/13

Número de embeddings: 13
Dimensión del vector: 3072
Primeros valores: [-0.014951821, -0.011913938, -0.00465584, 0.024265716, 0.031082446, -0.002729676, -0.011271667, -0.021816073, -0.005014977, -0.052264087]


### Una observación importante

Tenemos ahora dos representaciones del mismo chunk:

```text
Texto:
"Artículo IV. Incentivo..."

Embedding:
[0.01, -0.03, 0.17, ...]
```

El texto es lo que finalmente leerá el LLM.

El embedding es lo que utiliza el motor de búsqueda semántica.


## 9. Crear la base vectorial ChromaDB

Ahora construimos el índice.

ChromaDB almacenará conjuntamente:

```text
ID + embedding + texto + metadata
```

Primero creamos un cliente persistente y después una colección.

La carpeta `chroma_renting_db/` quedará en disco para poder reutilizarla posteriormente.


In [11]:
CHROMA_DIR.mkdir(parents=True, exist_ok=True)

chroma_client = chromadb.PersistentClient(
    path=str(CHROMA_DIR)
)

COLLECTION_NAME = "renting_documentos"

# Para que el notebook sea fácil de ejecutar varias veces,
# eliminamos la colección anterior si existe.
try:
    chroma_client.delete_collection(COLLECTION_NAME)
    print("Colección anterior eliminada.")
except Exception:
    pass

collection = chroma_client.create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"},
)

print("Colección creada:", COLLECTION_NAME)


Colección creada: renting_documentos


## 10. Insertar los chunks en ChromaDB

Ahora añadimos los datos al índice.

Chroma recibe cuatro cosas relacionadas por posición:

- `ids`
- `embeddings`
- `documents`
- `metadatas`

Es decir:

```text
chunk_0
   ├── embedding
   ├── texto
   └── metadata

chunk_1
   ├── embedding
   ├── texto
   └── metadata
```


In [12]:
ids = [item["id"] for item in chunks_preparados]

documents_for_chroma = [item["text"] for item in chunks_preparados]

metadatas = [item["metadata"] for item in chunks_preparados]

print("Chunks:", len(chunks_preparados))
print("IDs:", len(ids))
print("Documents:", len(documents_for_chroma))
print("Metadatas:", len(metadatas))
print("Embeddings:", len(embeddings))
print("Dimensión del primer embedding:", len(embeddings[0]))

Chunks: 13
IDs: 13
Documents: 13
Metadatas: 13
Embeddings: 13
Dimensión del primer embedding: 3072


In [13]:
ids = [item["id"] for item in chunks_preparados]
documents_for_chroma = [item["text"] for item in chunks_preparados]
metadatas = [item["metadata"] for item in chunks_preparados]

collection.add(
    ids=ids,
    embeddings=embeddings,
    documents=documents_for_chroma,
    metadatas=metadatas,
)

print("Documentos indexados:", collection.count())


Documentos indexados: 13


## 11. Comprobar el índice

Hasta aquí hemos terminado el **procesamiento offline**.

```text
PDF → chunks → embeddings → ChromaDB
```

La pregunta del usuario todavía no ha sido procesada.


In [14]:
print("ChromaDB:", CHROMA_DIR.resolve())
print("Colección:", collection.name)
print("Documentos indexados:", collection.count())


ChromaDB: C:\Users\Rubén\Documents\01-Estudios\Bootcamp IA\00 - Repos\02 - Camp AI\01 - The Bridge\05_RAG_Engineering\Sprint_10\Practica_live_review\EJERCICIO_LR_2\chroma_renting_db
Colección: renting_documentos
Documentos indexados: 13


# PARTE ONLINE: consultar el RAG

A partir de ahora simulamos lo que ocurriría cuando un usuario escribe una pregunta.

La diferencia fundamental es:

- **Offline:** procesamos los documentos.
- **Online:** procesamos la pregunta.


## 12. Escribir una pregunta

Empezamos con una pregunta que sí debería poder responderse con el documento.

El PDF contiene información sobre la campaña de Renting Tecnológico, incluyendo el periodo de adhesión, las condiciones y los incentivos. fileciteturn21file0turn21file2


In [15]:
pregunta = "¿Cuánto dura el contrato de Renting Tecnológico?"
print("Pregunta:", pregunta)


Pregunta: ¿Cuánto dura el contrato de Renting Tecnológico?


## 13. Convertir la pregunta en embedding

La pregunta pasa por **el mismo modelo de embeddings** utilizado para los documentos.

```text
Pregunta
   ↓
gemini-embedding-2
   ↓
vector de la pregunta
```

Ahora ese vector vive en el mismo espacio vectorial que los embeddings almacenados en ChromaDB.


In [16]:
query_response = client.models.embed_content(
    model=EMBEDDING_MODEL,
    contents=[pregunta],
)

query_embedding = query_response.embeddings[0].values

print("Dimensión:", len(query_embedding))
print("Primeros valores:", query_embedding[:10])


Dimensión: 3072
Primeros valores: [-0.020186769, -0.001215156, 0.0017450893, 0.016528435, 0.016775778, -0.017205115, 0.019190613, 0.0069136517, 0.014955572, -0.03752292]


## 14. Similarity Search

Ahora Chroma compara el embedding de la pregunta con los embeddings almacenados.

Pedimos los `TOP_K` chunks más cercanos.

Con distancia coseno:

> **menor distancia → mayor similitud**


In [17]:
resultado = collection.query(
    query_embeddings=[query_embedding],
    n_results=min(TOP_K, collection.count()),
    include=["documents", "metadatas", "distances"],
)

print("Claves devueltas:", resultado.keys())


Claves devueltas: dict_keys(['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas', 'distances'])


## 15. Inspeccionar el Top-K

Esta celda es especialmente importante.

**Todavía no hemos llamado al LLM.**

Estamos comprobando qué información ha recuperado el sistema.

```text
Pregunta
   ↓
Embedding
   ↓
ChromaDB
   ↓
Top-K chunks
```


In [18]:
for i in range(len(resultado["documents"][0])):
    documento = resultado["documents"][0][i]
    metadata = resultado["metadatas"][0][i]
    distancia = resultado["distances"][0][i]

    print("=" * 80)
    print(f"RESULTADO #{i + 1}")
    print(f"Distancia: {distancia:.4f}")
    print(f"Fuente: {metadata.get('source')}")
    print(f"Página: {metadata.get('page')}")
    print()
    print(documento)


RESULTADO #1
Distancia: 0.2916
Fuente: Renting.pdf
Página: 0

Renting:
*Renting ofrecido por Banco Iberico. Operación de renting sujeta a previa aprobación por parte del Banco. Oferta válida en
Península y Baleares hasta el 30/10/2026. Oferta no válida para Ceuta y Melilla.
**Renting ofrecido por Banco Iberico. Renta mensual del iPhone 17e 256 GB Sin Seguro 20,99€ €/mes (IVA incluido). Se
recibirá una bonificación de 20,99 € netos mensuales (tras aplicar la retención según normativa fiscal vigente, actualmente
el 19 %) por la contratación de un renting tecnológico a 36 meses para personas físicas que (1) domicilien por primera vez
su nómina o pensión superior a 1.200 € o cuota de autónomos o mutualidad y (2) la mantengan junto con la domiciliación
de dos recibos mensuales, (3) un movimiento mensual de tarjeta de crédito o saldo en cuenta igual o superior a 1.000 €
todos los días del mes y (4) tengan Bizum activo en Banco Iberico. Se recibirá una bonificación de 27,99 € netos mensuales


### Pregunta para el alumno

Antes de continuar, observa los resultados:

- ¿El chunk #1 parece responder la pregunta?
- ¿Los siguientes chunks aportan información adicional?
- ¿Qué pasa si aumentamos `TOP_K` de 3 a 5?
- ¿La menor distancia significa necesariamente que el chunk contiene toda la respuesta?

**Retrieval no significa respuesta.** Retrieval significa seleccionar candidatos relevantes.


## 16. Construir el contexto

El LLM no recibe directamente el objeto interno de ChromaDB.

Construimos un texto que contiene los chunks recuperados y su fuente.

```text
Top-K
  ↓
contexto textual
```


In [19]:
partes = []

for i in range(len(resultado["documents"][0])):
    documento = resultado["documents"][0][i]
    metadata = resultado["metadatas"][0][i]

    partes.append(
        f"--- Fragmento {i + 1} ---\n"
        f"Fuente: {metadata.get('source', 'desconocida')}\n"
        f"Página: {metadata.get('page', '?')}\n"
        f"{documento}"
    )

contexto = "\n\n".join(partes)

print(contexto)


--- Fragmento 1 ---
Fuente: Renting.pdf
Página: 0
Renting:
*Renting ofrecido por Banco Iberico. Operación de renting sujeta a previa aprobación por parte del Banco. Oferta válida en
Península y Baleares hasta el 30/10/2026. Oferta no válida para Ceuta y Melilla.
**Renting ofrecido por Banco Iberico. Renta mensual del iPhone 17e 256 GB Sin Seguro 20,99€ €/mes (IVA incluido). Se
recibirá una bonificación de 20,99 € netos mensuales (tras aplicar la retención según normativa fiscal vigente, actualmente
el 19 %) por la contratación de un renting tecnológico a 36 meses para personas físicas que (1) domicilien por primera vez
su nómina o pensión superior a 1.200 € o cuota de autónomos o mutualidad y (2) la mantengan junto con la domiciliación
de dos recibos mensuales, (3) un movimiento mensual de tarjeta de crédito o saldo en cuenta igual o superior a 1.000 €
todos los días del mes y (4) tengan Bizum activo en Banco Iberico. Se recibirá una bonificación de 27,99 € netos mensuales
(tras aplica

## 17. Construir el prompt RAG

Aquí aparece la parte de **Augmented Generation**.

El prompt combina:

1. instrucciones de comportamiento;
2. contexto recuperado;
3. pregunta del usuario.

La instrucción de abstención es importante: el modelo debe utilizar el contexto y reconocer cuando el contexto no contiene información suficiente.


In [20]:
INSTRUCCIONES_RAG = """
Eres un asistente que responde preguntas utilizando exclusivamente
la información del documento proporcionado como contexto.

Reglas:
- Responde basándote únicamente en el contexto.
- No inventes datos.
- Si el contexto no contiene información suficiente para responder,
  dilo explícitamente.
- Si la pregunta no está relacionada con el documento,
  indica que no puedes responderla a partir de la base de conocimiento.
- Cuando sea posible, menciona la página o fuente de donde procede la información.
"""

prompt = f"""
{INSTRUCCIONES_RAG}

--- CONTEXTO RECUPERADO ---
{contexto}

--- PREGUNTA ---
{pregunta}

--- RESPUESTA ---
"""

print(prompt)




Eres un asistente que responde preguntas utilizando exclusivamente
la información del documento proporcionado como contexto.

Reglas:
- Responde basándote únicamente en el contexto.
- No inventes datos.
- Si el contexto no contiene información suficiente para responder,
  dilo explícitamente.
- Si la pregunta no está relacionada con el documento,
  indica que no puedes responderla a partir de la base de conocimiento.
- Cuando sea posible, menciona la página o fuente de donde procede la información.


--- CONTEXTO RECUPERADO ---
--- Fragmento 1 ---
Fuente: Renting.pdf
Página: 0
Renting:
*Renting ofrecido por Banco Iberico. Operación de renting sujeta a previa aprobación por parte del Banco. Oferta válida en
Península y Baleares hasta el 30/10/2026. Oferta no válida para Ceuta y Melilla.
**Renting ofrecido por Banco Iberico. Renta mensual del iPhone 17e 256 GB Sin Seguro 20,99€ €/mes (IVA incluido). Se
recibirá una bonificación de 20,99 € netos mensuales (tras aplicar la retención segú

## 18. Generation: llamar a Gemini

Ahora sí llegamos a **Generation**.

```text
Prompt
  ↓
Gemini
  ↓
Respuesta
```

Observa la diferencia:

- ChromaDB **recupera** información.
- Gemini **genera** una respuesta utilizando esa información.


In [21]:
response = client.models.generate_content(
    model=GEMINI_MODEL,
    contents=prompt,
    config={"temperature": 0.2},
)

respuesta = (response.text or "").strip()

print(respuesta)


El contrato de renting tecnológico tiene una duración de 36 meses (Fuente: Renting.pdf, página 0).


## 19. Construir una función RAG completa

Ya entendimos cada paso por separado.

Ahora los encapsulamos en una única función:

```text
pregunta
   ↓
embedding
   ↓
ChromaDB
   ↓
Top-K
   ↓
contexto
   ↓
prompt
   ↓
Gemini
   ↓
respuesta
```

Esta función representa conceptualmente lo que en el proyecto Sprint 10 estaba encapsulado en `logic.py`.


In [22]:
def responder(pregunta: str, top_k: int = 3) -> dict:
    # 1. Embedding de la pregunta
    query_response = client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=[pregunta],
    )
    query_embedding = query_response.embeddings[0].values

    # 2. Retrieval
    resultado = collection.query(
        query_embeddings=[query_embedding],
        n_results=min(top_k, collection.count()),
        include=["documents", "metadatas", "distances"],
    )

    # 3. Construir contexto
    partes = []

    for i in range(len(resultado["documents"][0])):
        documento = resultado["documents"][0][i]
        metadata = resultado["metadatas"][0][i]

        partes.append(
            f"--- Fragmento {i + 1} ---\n"
            f"Fuente: {metadata.get('source', 'desconocida')}\n"
            f"Página: {metadata.get('page', '?')}\n"
            f"{documento}"
        )

    contexto = "\n\n".join(partes)

    # 4. Construir prompt
    prompt = f"""
{INSTRUCCIONES_RAG}

--- CONTEXTO RECUPERADO ---
{contexto}

--- PREGUNTA ---
{pregunta}

--- RESPUESTA ---
"""

    # 5. Generation
    response = client.models.generate_content(
        model=GEMINI_MODEL,
        contents=prompt,
        config={"temperature": 0.2},
    )

    respuesta = (response.text or "").strip()

    # 6. Fuentes
    fuentes = []
    for metadata in resultado["metadatas"][0]:
        fuente = metadata.get("source", "desconocida")
        if fuente not in fuentes:
            fuentes.append(fuente)

    return {
        "respuesta": respuesta,
        "fuentes": fuentes,
        "distancias": resultado["distances"][0],
        "contexto": contexto,
    }


## 20. Probar el RAG completo

Ahora una única llamada ejecuta todo el flujo online.


In [23]:
resultado_rag = responder(
    "¿Cuánto dura el contrato de Renting Tecnológico?",
    top_k=3,
)

print("RESPUESTA:")
print(resultado_rag["respuesta"])

print()
print("FUENTES:")
for fuente in resultado_rag["fuentes"]:
    print("-", fuente)


RESPUESTA:
El contrato de renting tecnológico tiene una duración de 36 meses (Fuente: Renting.pdf, página 0).

FUENTES:
- Renting.pdf


## 21. Experimento: cambiar la pregunta

Prueba preguntas que estén claramente soportadas por el documento.

El PDF indica un periodo de adhesión del **10 de abril de 2026 al 30 de octubre de 2026** y diferentes condiciones para acceder a los incentivos. fileciteturn21file0turn21file2


In [24]:
preguntas = [
    "¿Cuál es el periodo para adherirse a la campaña?",
    "¿Qué condiciones debe cumplir una persona física para recibir el incentivo?",
    "¿Cuánto es la bonificación para una nómina igual o superior a 3.000 euros?",
]

for pregunta in preguntas:
    resultado = responder(pregunta, top_k=3)

    print("=" * 80)
    print("PREGUNTA:", pregunta)
    print()
    print("RESPUESTA:", resultado["respuesta"])
    print()
    print("FUENTES:", resultado["fuentes"])


PREGUNTA: ¿Cuál es el periodo para adherirse a la campaña?

RESPUESTA: El periodo para adherirse a la campaña es del 10 de abril de 2026 al 30 de octubre de 2026, ambos inclusive (Fuente: Renting.pdf, página 2, Artículo II).

FUENTES: ['Renting.pdf']
PREGUNTA: ¿Qué condiciones debe cumplir una persona física para recibir el incentivo?

RESPUESTA: Para recibir el incentivo, una persona física (excepto autónomos) debe cumplir con las condiciones establecidas en el documento *Renting.pdf*, las cuales deben satisfacerse de forma simultánea al menos una vez dentro de los ocho meses posteriores a la firma de los documentos de adhesión:

1.  **Uso de tarjeta de crédito:** Realizar al menos un movimiento al mes con una tarjeta de crédito emitida por el Banco Ibérico de la que el participante sea titular. Los movimientos de tarjetas de débito no cuentan. Si no se cumple esta condición, la alternativa es mantener un saldo igual o superior a 1.000€ en la cuenta durante todos los días del mes en r

## 22. Experimento: variar Top-K

Ahora conectamos este ejercicio con lo aprendido sobre Retrieval.

Prueba:

```text
K = 1
K = 3
K = 5
```

Pregunta para discutir:

> ¿Más contexto siempre produce una mejor respuesta?

No necesariamente. Un `K` demasiado alto puede introducir ruido y aumentar el tamaño del prompt.


In [25]:
pregunta_experimento = "¿Qué condiciones debe cumplir una persona física para acceder al incentivo?"

for k in [1, 3, 5]:
    resultado = responder(pregunta_experimento, top_k=k)

    print("=" * 80)
    print("TOP-K:", k)
    print("RESPUESTA:")
    print(resultado["respuesta"])
    print("DISTANCIAS:", [round(d, 4) for d in resultado["distancias"]])


TOP-K: 1
RESPUESTA:
Para acceder al incentivo, una persona física (excepto autónomos) debe cumplir con las siguientes condiciones, según se detalla en la página 2 del documento *Renting.pdf*:

1.  **Uso de tarjeta de crédito:** Realizar al menos un movimiento al mes con una tarjeta de crédito emitida por el Banco Iberico de la cual sea titular. Los movimientos de tarjetas de débito no son válidos para este fin.
    *   *Nota:* La firma del Boletín de Adhesión no garantiza la aprobación de dicha tarjeta, ya que el Banco tiene su propio proceso de autorización.
    *   *Alternativa:* Si no se cumple la condición anterior, el cliente puede mantener un saldo igual o superior a 1.000€ en su cuenta durante todos los días del mes en revisión.
2.  **Servicio Bizum:** Estar dado de alta en el servicio de Bizum asociado a una cuenta en Banco Iberico.
DISTANCIAS: [0.2948]
TOP-K: 3
RESPUESTA:
Para acceder al incentivo, una persona física debe cumplir con las siguientes condiciones, según se detall

## 23. Experimento: una pregunta fuera del dominio

Ahora preguntamos algo que el documento no trata:

```text
¿Cuál es la capital de Francia?
```

La intención didáctica es comprobar que un RAG de dominio específico **no debería convertir automáticamente al LLM en un asistente de conocimiento general**.

La protección se consigue combinando retrieval, contexto y las instrucciones del prompt. No es una garantía matemática: un LLM puede todavía desviarse, por lo que en sistemas reales se suelen añadir validaciones adicionales.


In [26]:
resultado_fuera = responder(
    "¿Cuál es la capital de Francia?",
    top_k=3,
)

print("RESPUESTA:")
print(resultado_fuera["respuesta"])
print()
print("FUENTES RECUPERADAS:")
print(resultado_fuera["fuentes"])


RESPUESTA:
No puedo responder a esa pregunta a partir de la base de conocimiento proporcionada, ya que la información sobre la capital de Francia no está relacionada con el documento.

FUENTES RECUPERADAS:
['Renting.pdf']


## 24. Inspeccionar qué recuperó ChromaDB para una pregunta indebida

Este experimento es muy útil para entender una propiedad fundamental del RAG.

Aunque la pregunta sea ajena al dominio, ChromaDB **siempre intentará encontrar los vectores más cercanos**.

Por eso:

```text
Pregunta fuera del dominio
          ↓
     embedding
          ↓
       ChromaDB
          ↓
    algunos chunks
```

El hecho de recuperar chunks **no significa que exista una respuesta válida**.

La generación debe decidir, siguiendo las instrucciones, si el contexto realmente permite responder.


In [27]:
pregunta_fuera = "¿Cuál es la capital de Francia?"

query_response = client.models.embed_content(
    model=EMBEDDING_MODEL,
    contents=[pregunta_fuera],
)

fuera_embedding = query_response.embeddings[0].values

resultado_fuera_retrieval = collection.query(
    query_embeddings=[fuera_embedding],
    n_results=TOP_K,
    include=["documents", "metadatas", "distances"],
)

for i in range(len(resultado_fuera_retrieval["documents"][0])):
    print("=" * 80)
    print(f"RESULTADO #{i + 1}")
    print("Distancia:", round(resultado_fuera_retrieval["distances"][0][i], 4))
    print("Página:", resultado_fuera_retrieval["metadatas"][0][i].get("page"))
    print(resultado_fuera_retrieval["documents"][0][i][:700])


RESULTADO #1
Distancia: 0.5157
Página: 0
en support.apple.com/es-es/121115. 4. La pantalla tiene esquinas
redondeadas que siguen el elegante diseño curvo del teléfono, y las
esquinas se encuentran dentro de un rectángulo estándar. Si se
mide en forma de rectángulo estándar, la pantalla tiene 6.06
pulgadas (iPhone 17e), 6.27 pulgadas (iPhone 17, iPhone 17 Pro),
6.55 pulgadas (iPhone Air) o 6.86 pulgadas (iPhone 17 Pro Max) en
diagonal. El área real de visualización es menor. 5. Pruebas
realizadas por Apple en julio de 2025 con prototipos del iPhone 17,
iPhone Air, iPhone 17 Pro y iPhone 17 Pro Max y versiones
preliminares de software empleando el cable de carga USB C con el
adaptador de Apple con potencia dinámica de 40 a 60 W máx.

RESULTADO #2
Distancia: 0.5283
Página: 0
cuenta igual o superior a 1.000 € todos los días del mes y (4) tengan Bizum activo en Banco Iberico. No obstante lo anterior,
el importe de la bonificación recibida será como máximo el importe de la renta del renting 

## 25. Arquitectura completa

### OFFLINE

```text
Renting.pdf
    ↓
PyPDFLoader
    ↓
Documentos
    ↓
Chunking
    ↓
Chunks
    ↓
Embeddings
    ↓
ChromaDB
```

### ONLINE

```text
Pregunta
    ↓
Embedding de pregunta
    ↓
Similarity Search
    ↓
Top-K
    ↓
Contexto
    ↓
Prompt
    ↓
Gemini
    ↓
Respuesta
```

Esta separación es fundamental en una arquitectura RAG.


## 26. Relación con el proyecto Sprint 10

Lo que hemos hecho manualmente en este notebook corresponde conceptualmente a los módulos del proyecto:

| Notebook | Proyecto Sprint 10 |
|---|---|
| Carga del PDF | `load.py` |
| Chunking | `chunk.py` |
| Embeddings | `embed.py` |
| Crear/consultar ChromaDB | `index.py` + `retriever.py` |
| Construcción del contexto | `context.py` |
| Construcción del prompt | `prompts.py` |
| Llamada a Gemini | `generate.py` |
| Orquestación completa | `logic.py` |

La ventaja del notebook es didáctica: aquí podemos ver cada transformación de datos antes de encapsularla en módulos.


# 27. Ejercicio final para los alumnos

Completa estos experimentos.

### A. Chunking

Cambia:

```python
CHUNK_SIZE = 1500
CHUNK_OVERLAP = 200
```

por otros valores y observa cómo cambia el número y contenido de los chunks.

### B. Retrieval

Prueba diferentes valores de `TOP_K`.

### C. Preguntas

Prueba al menos:

1. Una pregunta sobre el periodo de la campaña.
2. Una pregunta sobre los requisitos.
3. Una pregunta sobre los incentivos.
4. Una pregunta que combine dos condiciones.
5. Una pregunta completamente ajena al documento.

### D. Análisis

Para cada pregunta intenta responder:

- ¿Qué chunks recuperó ChromaDB?
- ¿Qué distancia tenían?
- ¿El contexto contenía realmente la respuesta?
- ¿Gemini respondió solo con la información recuperada?
- ¿Cuándo debería abstenerse el sistema?

**Objetivo:** no evaluar solamente si "la respuesta parece buena". Evaluar todo el pipeline.


# Resumen

Hemos construido un RAG completo desde cero:

```text
                 OFFLINE
                 ───────
              Renting.pdf
                   ↓
                Chunks
                   ↓
               Embeddings
                   ↓
                ChromaDB
                   │
                   ▼
                 ONLINE
                 ──────
             Pregunta usuario
                   ↓
          Embedding pregunta
                   ↓
          Similarity Search
                   ↓
                  Top-K
                   ↓
                Contexto
                   ↓
                 Prompt
                   ↓
                 Gemini
                   ↓
               Respuesta
```

**RAG = Retrieval + Augmented Generation**

La idea central:

> **ChromaDB recupera conocimiento; Gemini genera una respuesta utilizando ese conocimiento.**
